# Exploratory Data Analysis — E-Commerce Pipeline

Notebook for ad-hoc exploration of the Bronze/Silver/Gold datasets.
Run `python pipeline/run_all.py` before executing cells.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

from config.settings import settings

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Bronze Layer — Raw Data Overview

In [ ]:
orders = pd.read_csv(settings.RAW_DATA_PATH / 'orders.csv', parse_dates=['order_date'])
customers = pd.read_csv(settings.RAW_DATA_PATH / 'customers.csv')
products = pd.read_csv(settings.RAW_DATA_PATH / 'products.csv')
items = pd.read_csv(settings.RAW_DATA_PATH / 'order_items.csv')
payments = pd.read_csv(settings.RAW_DATA_PATH / 'payments.csv')
reviews = pd.read_csv(settings.RAW_DATA_PATH / 'reviews.csv')

for name, df in [('orders', orders), ('customers', customers), ('products', products),
                  ('items', items), ('payments', payments), ('reviews', reviews)]:
    print(f'{name:15s}  rows={len(df):>6,}  cols={df.shape[1]}')

In [ ]:
# Null rates per column — orders
(orders.isnull().mean() * 100).round(2).sort_values(ascending=False)

## 2. Order Volume Over Time

In [ ]:
monthly = (
    orders
    .assign(year_month=orders['order_date'].dt.to_period('M'))
    .groupby('year_month')
    .agg(order_count=('order_id', 'count'), revenue=('total_amount', 'sum'))
    .reset_index()
)
monthly['year_month_str'] = monthly['year_month'].astype(str)

fig, axes = plt.subplots(1, 2)
axes[0].bar(monthly['year_month_str'], monthly['order_count'], color='steelblue')
axes[0].set_title('Monthly Order Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(monthly['year_month_str'], monthly['revenue'], color='darkorange')
axes[1].set_title('Monthly Revenue (R$)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))

plt.tight_layout()
plt.show()

## 3. Revenue by Category

In [ ]:
enriched = (
    items
    .merge(products[['product_id', 'category']], on='product_id')
    .merge(orders[['order_id', 'status']], on='order_id')
    .query("status == 'delivered'")
)
by_cat = enriched.groupby('category')['line_total'].sum().sort_values(ascending=True)

by_cat.plot(kind='barh', color='teal')
plt.title('Revenue by Category (delivered orders)')
plt.xlabel('Revenue (R$)')
plt.tight_layout()
plt.show()

## 4. Rating Distribution

In [ ]:
reviews['rating'].value_counts().sort_index().plot(
    kind='bar', color='mediumpurple', rot=0
)
plt.title('Review Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 5. Payment Method Mix

In [ ]:
payments['method'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', startangle=90,
    colors=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']
)
plt.title('Payment Method Distribution')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 6. Data Quality Report Preview

In [ ]:
import json

report_path = settings.PROCESSED_DATA_PATH / 'quality_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(f"Overall: {report['overall_status']}  ({report['passed']}/{report['total_checks']} checks passed)")
    failed = [c for c in report['checks'] if c['status'] == 'FAIL']
    if failed:
        print(f"\nFailed checks ({len(failed)}):")
        for c in failed:
            print(f"  [{c['check']}] {c.get('dataset', '')} – errors: {c['error_count']}")
else:
    print('Quality report not found — run pipeline/run_all.py first.')

## 7. Gold Layer — Fact Table Sample

In [ ]:
fato_path = settings.ANALYTICS_DATA_PATH / 'fato_pedidos'
if fato_path.exists():
    fato = pd.read_parquet(fato_path)
    print(f'fato_pedidos: {len(fato):,} rows')
    display(fato.head())
    display(fato.describe())
else:
    print('Gold layer not found — run pipeline/run_all.py first.')